In [ ]:
# ════════════════════════════════════════════════════════════════
# MODE & CONFIG
# ════════════════════════════════════════════════════════════════

QUICK_TEST = False  # True = 5 steps only (verify pipeline), 
SMOKE_TEST = False  # True = small data 1 epoch , False = FULL 
USE_GRPO = False    # GRPO not helping yet — model needs more SFT first
USE_COT_DATA = True # Use konbu17's verified CoT data

BASE_MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'

if QUICK_TEST:
    print('>>> QUICK TEST MODE — 5 steps SFT + 5 steps GRPO')
elif SMOKE_TEST:
    print('SMOKE TEST MODE — small data, 1 epoch ')
else:
    print('FULL TRAINING MODE — all data, 2 epochs ')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 2: TRITON INSTALL + ENVIRONMENT FIXES
# ════════════════════════════════════════════════════════════════
import os, glob, sys, subprocess, site, stat, shutil

# ── Install Triton from offline wheel ──
candidates = glob.glob('/kaggle/input/**/*triton*.whl', recursive=True)
if candidates:
    wheel = candidates[0]
    target = '/kaggle/working/pydeps'
    os.makedirs(target, exist_ok=True)
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--no-deps',
         '--target', target, '--upgrade', '--ignore-installed', wheel],
        check=True, capture_output=True
    )
    if target not in sys.path:
        sys.path.insert(0, target)
    site.addsitedir(target)
    print(f'✅ Triton installed from {os.path.basename(wheel)}')
else:
    print('ℹ️ No Triton wheel found, using pre-installed')

# ── Utility script path ──
sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')

# ── ptxas-blackwell fix ──
ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
ptxas_dst = '/tmp/ptxas-blackwell'
if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
    shutil.copy2(ptxas_src, ptxas_dst)
    os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    src_bin = os.path.dirname(ptxas_src)
    dst_bin = '/tmp/triton_nvidia_bin'
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst
    try:
        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst
    except Exception:
        pass
    print('✅ ptxas-blackwell fix applied')
else:
    print('ℹ️ ptxas-blackwell not found, skipping')

try:
    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'
except Exception:
    pass

# ── CUDA memory config ──
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('✅ Environment fixes applied')


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 3: OFFLINE PACKAGE INSTALL (PROVEN APPROACH)
# ════════════════════════════════════════════════════════════════
import glob, importlib.util, subprocess, sys, types

def sh(cmd, check=True):
    print('+', cmd)
    return subprocess.run(cmd, shell=True, check=check, capture_output=True, text=True)

def find_spec(name):
    return importlib.util.find_spec(name) is not None

def recursive_wheels(pattern):
    return sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True))

import torch
py_tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
torch_mm = '.'.join(torch.__version__.split('+')[0].split('.')[:2])
abi_tag = 'cxx11abiTRUE' if torch.compiled_with_cxx11_abi() else 'cxx11abiFALSE'
print(f'Python: {sys.version_info.major}.{sys.version_info.minor} | '
      f'Torch: {torch.__version__} | CUDA: {torch.version.cuda}')

def pick_best(wheels):
    exact = [w for w in wheels if py_tag in w and f'torch{torch_mm}' in w and abi_tag in w]
    if exact: return exact[-1]
    py_only = [w for w in wheels if py_tag in w]
    if py_only: return py_only[-1]
    return wheels[-1] if wheels else None

# ── Install trl from offline packages ──
if not find_spec('trl'):
    offline_dirs = [
        '/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/',
        '/kaggle/input/notebooks/amanatar/nemotron-offline-packages/offline_packages/',
    ]
    for odir in offline_dirs:
        if os.path.exists(odir):
            r = sh(f'pip install --no-index --find-links={odir} trl', check=False)
            if r.returncode == 0:
                break
    if not find_spec('trl'):
        sh('pip install trl', check=False)

# ── Install mamba_ssm + causal_conv1d from pre-built wheels ──
all_mamba = recursive_wheels('mamba_ssm-*.whl')
all_causal = recursive_wheels('causal*conv1d*.whl')
print(f'Found mamba wheels: {len(all_mamba)} | causal_conv1d wheels: {len(all_causal)}')

if not find_spec('causal_conv1d') and all_causal:
    w = pick_best(all_causal)
    if w:
        sh(f'{sys.executable} -m pip install --no-index --no-deps "{w}"')

if not find_spec('mamba_ssm') and all_mamba:
    w = pick_best(all_mamba)
    if w:
        sh(f'{sys.executable} -m pip install --no-index --no-deps "{w}"')

# ── Stub mamba3 modules (needs cutlass which isn't available) ──
for _mod_name in [
    'mamba_ssm.modules.mamba3',
    'mamba_ssm.ops.cute',
    'mamba_ssm.ops.cute.mamba3',
    'mamba_ssm.ops.cute.mamba3.mamba3_step_fn',
]:
    _m = types.ModuleType(_mod_name)
    _m.__path__ = []
    _m.__package__ = _mod_name
    sys.modules[_mod_name] = _m
sys.modules['mamba_ssm.modules.mamba3'].Mamba3 = None

# ── Verify ──
import mamba_ssm
import datasets
import trl
print(f'✅ mamba_ssm: {mamba_ssm.__version__}')
print(f'✅ datasets:  {datasets.__version__}')
print(f'✅ trl:       {trl.__version__}')


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 4: ALL IMPORTS
# ════════════════════════════════════════════════════════════════
import os, sys, gc, re, math, json, time, zipfile, warnings, shutil
warnings.filterwarnings('ignore')

import torch
import torch.nn.functional as F
import polars as pl
import kagglehub
from collections import defaultdict
from decimal import Decimal, ROUND_HALF_UP
from itertools import combinations

from datasets import Dataset as HFDataset
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback
from trl import SFTTrainer, SFTConfig, GRPOTrainer, GRPOConfig

print(f'✅ All imports OK | PyTorch {torch.__version__} | '
      f'CUDA {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 5: CONFIGURATION (MATCHED TO COMPETITION EVAL RULES)
# ════════════════════════════════════════════════════════════════
# Competition eval: max_lora_rank=32, max_tokens=7680, temperature=0.0
# We MUST train with these constraints!

if QUICK_TEST:
    SFT_EPOCHS = 1
    SFT_SUBSAMPLE = 50
    GRPO_SUBSAMPLE = 20
    GRPO_EPOCHS = 1
    MAX_STEPS_SFT = 5
    MAX_STEPS_GRPO = 5
elif SMOKE_TEST:
    SFT_EPOCHS = 1
    SFT_SUBSAMPLE = 800
    GRPO_SUBSAMPLE = 100
    GRPO_EPOCHS = 1
    MAX_STEPS_SFT = -1
    MAX_STEPS_GRPO = 30
else:
    # ── FULL TRAINING ──
    SFT_EPOCHS = 1            # Proven: 2 epochs
    SFT_SUBSAMPLE = 0         # 0 = ALL data
    GRPO_SUBSAMPLE = 100
    GRPO_EPOCHS = 1
    MAX_STEPS_SFT = -1
    MAX_STEPS_GRPO = 30

# LoRA — CANNOT exceed rank 32 (competition rule!)
LORA_RANK = 32             # MAX ALLOWED by competition
LORA_ALPHA = 32            # Standard: alpha = rank
LORA_DROPOUT = 0.0         # 0 at eval → train with 0 too (consistency)
USE_RSLORA = False

# SFT
SFT_LR = 1e-4              # Proven by konbu17
SFT_BATCH = 1
SFT_GRAD_ACCUM = 8         # Effective batch = 8
SFT_MAX_LEN = 7680         # MATCH EVAL max_tokens=7680 (was 4096 — truncating CoT!)
SFT_PACKING = False
NEFTUNE_ALPHA = 5.0         # Regularization during training

# GRPO (if enabled)
GRPO_LR = 5e-6
GRPO_NUM_GENERATIONS = 4
GRPO_MAX_COMPLETION = 1024
GRPO_TEMPERATURE = 0.8
GRPO_BETA = 0.04

OUTPUT_DIR = '/kaggle/working/sft_adapter'
os.makedirs(OUTPUT_DIR, exist_ok=True)

PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

print(f'Config: epochs={SFT_EPOCHS}, lr={SFT_LR}, rank={LORA_RANK}, max_len={SFT_MAX_LEN}')
print(f'SFT subsample={SFT_SUBSAMPLE} (0=all), GRPO={USE_GRPO}, dropout={LORA_DROPOUT}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 6: DATA LOADING + PUZZLE CLASSIFICATION
# ════════════════════════════════════════════════════════════════
import pandas as pd

# ── Try CoT data first (konbu17's verified-correct reasoning) ──
COT_PATH = '/kaggle/input/datasets/konbu17/nemotron-sft-lora-cot-selection/train_split_with_cot.csv'
TRAIN_PATH = '/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv'

if USE_COT_DATA and os.path.exists(COT_PATH):
    cot_df = pd.read_csv(COT_PATH)
    print(f'✅ CoT data loaded: {len(cot_df)} verified-correct samples')
    print(cot_df['type'].value_counts().sort_index())
    has_cot = True
else:
    has_cot = False
    print('ℹ️ No CoT data — using competition train.csv with our solvers')

# Always load competition data (for GRPO and fallback)
train_full = pl.read_csv(TRAIN_PATH)
print(f'\nCompetition train data: {len(train_full)} samples')

def classify_puzzle(prompt):
    p = prompt.lower()
    if re.search(r'numeral system|base[- ]?\d|number.*convert|radix|secret number|roman', p):
        return 'Number Base Conversion'
    elif re.search(r'gravit|gravity|falling|free.?fall|acceleration due to', p):
        return 'Gravitational Constant'
    elif re.search(r'transformation rule|equation.*transform|secret.*rule.*equation|rule.*applied.*equation', p):
        return 'Equation Transformation'
    elif re.search(r'encrypt|cipher|secret.*code.*letter|coded.*message|secret.*text|decrypt', p):
        return 'Text Encryption'
    elif re.search(r'bit.?manipul|binary|8.?bit|bitwise|bit.*transform', p):
        return 'Bit Manipulation'
    elif re.search(r'unit.?conver|measurement|becomes.*\d|secret.*conver.*measur', p):
        return 'Unit Conversion'
    else:
        return 'Unknown'

train_full = train_full.with_columns(
    pl.col('prompt').map_elements(classify_puzzle, return_dtype=pl.Utf8).alias('puzzle_type')
)
print('\nPuzzle type distribution:')
print(train_full.group_by('puzzle_type').agg(pl.len().alias('count')).sort('count', descending=True))


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 7: MODEL LOADING + LoRA
# ════════════════════════════════════════════════════════════════

MODEL_PATH = kagglehub.model_download('metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')
print(f'Model path: {MODEL_PATH}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, device_map='auto', trust_remote_code=True, dtype=torch.bfloat16,
)
print('✅ Model loaded')

# ── LoRA ──
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=r'.*\.(in_proj|out_proj|up_proj|down_proj)$',
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
    use_rslora=USE_RSLORA,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# CELL 5: ALL DETERMINISTIC SOLVERS (ENHANCED)
# ════════════════════════════════════════════════════════════════════════════════

BOXED_INSTRUCTION = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

def _round2_candidates(val):
    candidates = set()
    candidates.add(f"{round(val, 2):.2f}")
    d = Decimal(str(val)).quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)
    candidates.add(str(d))
    candidates.add(f"{math.floor(val * 100) / 100:.2f}")
    candidates.add(f"{math.ceil(val * 100) / 100:.2f}")
    for c in list(candidates):
        if c.endswith('0') and '.' in c:
            candidates.add(c.rstrip('0').rstrip('.'))
    return candidates

# ── All solver functions (gravity, unit, base, encryption, bit, equation) ──
# [SAME as original v1 solvers - kept identical for reliability]

def solve_gravity(prompt, answer):
    pairs = re.findall(r't\s*=\s*([\d.]+)\s*s.*?distance\s*=\s*([\d.]+)\s*m', prompt)
    query_t_m = re.search(r'falling distance for t\s*=\s*([\d.]+)\s*s', prompt)
    if not pairs or not query_t_m:
        return None, None
    gs = [2 * float(d) / (float(t)**2) for t, d in pairs]
    g_avg = sum(gs) / len(gs)
    t_q = float(query_t_m.group(1))
    best_g = g_avg
    val = 0.5 * g_avg * t_q**2
    candidates = _round2_candidates(val)
    if answer not in candidates:
        for gi in gs:
            vi = 0.5 * gi * t_q**2
            if answer in _round2_candidates(vi):
                best_g, val = gi, vi
                break
    predicted = answer if answer in _round2_candidates(val) else f"{val:.2f}"
    g_lines = "\n".join(f"  Example {i+1}: g = 2 * {d} / {t}^2 = {2*float(d)/float(t)**2:.4f}"
                        for i, (t, d) in enumerate(pairs))
    cot = (f"I need to find the hidden gravitational constant g from the examples using d = 0.5 * g * t^2, so g = 2d / t^2.\n\n"
           f"Computing g from each example:\n{g_lines}\n\n"
           f"Average g = {best_g:.4f}\n\n"
           f"For t = {t_q}s:\nd = 0.5 * {best_g:.4f} * {t_q}^2 = {predicted}\n\n"
           f"\\boxed{{{predicted}}}")
    return predicted, cot

def solve_unit_conversion(prompt, answer):
    pairs = re.findall(r'([\d.]+)\s*m\s+becomes\s+([\d.]+)', prompt)
    query_m = re.search(r'convert the following measurement:\s*([\d.]+)\s*m', prompt)
    if not pairs or not query_m:
        return None, None
    ratios = [float(out) / float(inp) for inp, out in pairs]
    ratio = sum(ratios) / len(ratios)
    q = float(query_m.group(1))
    val = ratio * q
    predicted = f"{val:.2f}"
    if answer not in _round2_candidates(val):
        for r in ratios:
            if answer in _round2_candidates(r * q):
                ratio, val = r, r * q
                break
    if answer in _round2_candidates(val):
        predicted = answer
    ratio_lines = "\n".join(f"  Example {i+1}: {out} / {inp} = {float(out)/float(inp):.4f}"
                            for i, (inp, out) in enumerate(pairs))
    cot = (f"I need to find the secret conversion factor from the examples.\n\n"
           f"Computing ratio for each:\n{ratio_lines}\n\n"
           f"Factor = {ratio:.4f}\nFor input {q}m: Result = {ratio:.4f} * {q} = {predicted}\n\n"
           f"\\boxed{{{predicted}}}")
    return predicted, cot

def _int_to_roman(num):
    vals = [1000, 900, 500, 400, 100, 90, 50, 40, 10, 9, 5, 4, 1]
    syms = ['M', 'CM', 'D', 'CD', 'C', 'XC', 'L', 'XL', 'X', 'IX', 'V', 'IV', 'I']
    result = ''
    for v, s in zip(vals, syms):
        while num >= v: result += s; num -= v
    return result

def _roman_to_int(s):
    vals = {'I':1,'V':5,'X':10,'L':50,'C':100,'D':500,'M':1000}
    total = 0
    for i in range(len(s)):
        if i+1 < len(s) and vals.get(s[i],0) < vals.get(s[i+1],0):
            total -= vals.get(s[i],0)
        else:
            total += vals.get(s[i],0)
    return total

def solve_base_conversion(prompt, answer):
    # Decimal -> Roman
    examples = re.findall(r'(\d+)\s*->\s*([A-Z]+)', prompt)
    if examples:
        is_roman = all(_int_to_roman(int(n)) == r for n, r in examples)
        if is_roman:
            query = re.search(r'(?:write|convert)\s+(?:the\s+)?number\s+(\d+)', prompt)
            if query:
                num = int(query.group(1))
                predicted = _int_to_roman(num)
                remainder = num; parts = []
                vals = [1000,900,500,400,100,90,50,40,10,9,5,4,1]
                syms = ['M','CM','D','CD','C','XC','L','XL','X','IX','V','IV','I']
                for v,s in zip(vals,syms):
                    while remainder >= v: parts.append(f"{s}({v})"); remainder -= v
                cot = (f"Decimal to Roman conversion.\n"
                       f"Examples verify: {', '.join(f'{n}->{r}' for n,r in examples[:3])}\n"
                       f"{num} = {' + '.join(parts)} = {predicted}\n\n\\boxed{{{predicted}}}")
                return predicted, cot
    # Roman -> Decimal
    examples_rev = re.findall(r'([A-Z]+)\s*->\s*(\d+)', prompt)
    if examples_rev:
        is_roman = all(_roman_to_int(r) == int(n) for r, n in examples_rev)
        if is_roman:
            query = re.search(r'(?:write|convert)\s+(?:the\s+)?(?:number\s+)?([A-Z]+)', prompt)
            if query:
                rom = query.group(1)
                predicted = str(_roman_to_int(rom))
                cot = (f"Roman to decimal: {rom}\n"
                       f"{'  '.join(f'{c}={_roman_to_int(c)}' for c in rom)}\n"
                       f"Total = {predicted}\n\n\\boxed{{{predicted}}}")
                return predicted, cot
    # NEW: Arbitrary base conversion detection
    base_examples = re.findall(r'(\d+)\s*->\s*(\d+)', prompt)
    if len(base_examples) >= 2:
        # Try common bases: binary, octal, hex
        for from_base in [2,8,10,16]:
            for to_base in [2,8,10,16]:
                if from_base == to_base: continue
                try:
                    matches = all(
                        _convert_base(int(a), from_base, to_base) == b
                        for a, b in base_examples
                    )
                    if matches:
                        query = re.search(r'(?:write|convert)\s+(?:the\s+)?(?:number\s+)?(\d+)', prompt)
                        if query:
                            q = int(query.group(1))
                            predicted = _convert_base(q, from_base, to_base)
                            if predicted == answer:
                                cot = (f"Base {from_base} to base {to_base} conversion.\n"
                                       f"Converting {q}: {predicted}\n\n\\boxed{{{predicted}}}")
                                return predicted, cot
                except: pass
    return None, None

def _convert_base(num_str, from_base, to_base):
    """Convert number string from one base to another."""
    try:
        decimal_val = int(str(num_str), from_base)
        if to_base == 10: return str(decimal_val)
        if to_base == 2: return bin(decimal_val)[2:]
        if to_base == 8: return oct(decimal_val)[2:]
        if to_base == 16: return hex(decimal_val)[2:].upper()
        # General base
        if decimal_val == 0: return "0"
        digits = []
        while decimal_val > 0:
            digits.append(str(decimal_val % to_base))
            decimal_val //= to_base
        return ''.join(reversed(digits))
    except: return None

def solve_text_encryption(prompt, answer):
    is_decrypt = 'decrypt' in prompt.lower()
    examples = re.findall(r'(.+?)\s*->\s*(.+)', prompt)
    query_m = re.search(r'(?:de|en)crypt the following text:\s*(.+?)(?:\n|$)', prompt)
    if not examples or not query_m: return None, None
    query = query_m.group(1).strip()
    char_map = {}
    for a, b in examples:
        a, b = a.strip(), b.strip()
        if len(a) != len(b): continue
        for x, y in zip(a, b):
            if x == ' ' and y == ' ': continue
            if x in char_map and char_map[x] != y: return None, None
            char_map[x] = y
    if len(query) == len(answer):
        for c, p in zip(query, answer):
            if c == ' ' and p == ' ': continue
            if c in char_map and char_map[c] != p: return None, None
            char_map[c] = p
    result = ''
    for c in query:
        if c == ' ': result += ' '
        elif c in char_map: result += char_map[c]
        else: return None, None
    direction = "cipher→plain" if is_decrypt else "plain→cipher"
    shown = {c: char_map[c] for c in query if c != ' ' and c in char_map}
    table = ", ".join(f"'{k}'→'{v}'" for k,v in sorted(shown.items()))
    cot = (f"Substitution cipher ({direction}).\nMappings: {table}\n"
           f"'{query}' → '{result}'\n\n\\boxed{{{result}}}")
    return result, cot

# ── Bit Manipulation (enhanced with shift detection) ──
def _get_bit(s, pos): return int(s[pos])

def _solve_bit_functions(pairs):
    funcs = [None] * 8
    for out_pos in range(8):
        expected = [_get_bit(out, out_pos) for _, out in pairs]
        # Single-bit
        for in_pos in range(8):
            direct = [_get_bit(inp, in_pos) for inp, _ in pairs]
            if direct == expected: funcs[out_pos] = ('direct', in_pos); break
            if [1-b for b in direct] == expected: funcs[out_pos] = ('not', in_pos); break
        if funcs[out_pos]: continue
        # Constant bit (NEW)
        if all(b == 0 for b in expected): funcs[out_pos] = ('const', 0); continue
        if all(b == 1 for b in expected): funcs[out_pos] = ('const', 1); continue
        # Two-bit
        found = False
        for i, j in combinations(range(8), 2):
            bi = [_get_bit(inp, i) for inp, _ in pairs]
            bj = [_get_bit(inp, j) for inp, _ in pairs]
            tests = [('xor', [a^b for a,b in zip(bi,bj)]),
                     ('xnor', [1-(a^b) for a,b in zip(bi,bj)]),
                     ('and', [a&b for a,b in zip(bi,bj)]),
                     ('nand', [1-(a&b) for a,b in zip(bi,bj)]),
                     ('or', [a|b for a,b in zip(bi,bj)]),
                     ('nor', [1-(a|b) for a,b in zip(bi,bj)])]
            for name, result in tests:
                if result == expected: funcs[out_pos] = (name, i, j); found = True; break
            if found: break
        if funcs[out_pos]: continue
        # Three-bit
        for i, j, k in combinations(range(8), 3):
            bi = [_get_bit(inp, i) for inp, _ in pairs]
            bj = [_get_bit(inp, j) for inp, _ in pairs]
            bk = [_get_bit(inp, k) for inp, _ in pairs]
            tests = [('majority', [1 if (a+b+c)>=2 else 0 for a,b,c in zip(bi,bj,bk)]),
                     ('minority', [1 if (a+b+c)<2 else 0 for a,b,c in zip(bi,bj,bk)]),
                     ('choice', [b if a==1 else c for a,b,c in zip(bi,bj,bk)])]
            for name, result in tests:
                if result == expected: funcs[out_pos] = (name, i, j, k); found = True; break
            if found: break
    return funcs

def _apply_bit_func(func, query):
    if func is None: return None
    name = func[0]
    if name == 'const': return func[1]
    if name == 'direct': return _get_bit(query, func[1])
    if name == 'not': return 1 - _get_bit(query, func[1])
    if name in ('xor','xnor','and','nand','or','nor'):
        a, b = _get_bit(query, func[1]), _get_bit(query, func[2])
        ops = {'xor':a^b, 'xnor':1-(a^b), 'and':a&b, 'nand':1-(a&b), 'or':a|b, 'nor':1-(a|b)}
        return ops[name]
    if name in ('majority','minority','choice'):
        a,b,c = _get_bit(query,func[1]), _get_bit(query,func[2]), _get_bit(query,func[3])
        if name == 'majority': return 1 if (a+b+c)>=2 else 0
        if name == 'minority': return 1 if (a+b+c)<2 else 0
        if name == 'choice': return b if a==1 else c
    return None

def _describe_bit_func(func):
    if func is None: return "unknown"
    name = func[0]
    if name == 'const': return f"constant({func[1]})"
    if name == 'direct': return f"input[{func[1]}]"
    if name == 'not': return f"NOT input[{func[1]}]"
    if name in ('xor','xnor','and','nand','or','nor'):
        return f"input[{func[1]}] {name.upper()} input[{func[2]}]"
    if name in ('majority','minority'):
        return f"{name}(input[{func[1]}], input[{func[2]}], input[{func[3]}])"
    if name == 'choice':
        return f"if input[{func[1]}] then input[{func[2]}] else input[{func[3]}]"
    return str(func)

def solve_bit_manipulation(prompt, answer):
    pairs = re.findall(r'([01]{8})\s*->\s*([01]{8})', prompt)
    query_m = re.search(r'output for:\s*([01]{8})', prompt)
    if not pairs or not query_m: return None, None
    query = query_m.group(1)
    funcs = _solve_bit_functions(pairs)
    predicted_bits = [_apply_bit_func(f, query) for f in funcs]
    all_solved = all(b is not None for b in predicted_bits)
    if all_solved:
        predicted = ''.join(str(b) for b in predicted_bits)
        if predicted == answer:
            cot = "Finding the bit transformation rule.\n\n"
            for i in range(8):
                cot += f"  output[{i}] = {_describe_bit_func(funcs[i])} = {predicted_bits[i]}\n"
            cot += f"\nFor {query}: {predicted}\n\n\\boxed{{{predicted}}}"
            return predicted, cot
    # Scaffold with answer
    cot = f"Analyzing bit transformation from {len(pairs)} examples.\n\n"
    for i in range(8):
        desc = _describe_bit_func(funcs[i]) if funcs[i] else "complex function"
        cot += f"  output[{i}] = {desc}\n"
    cot += f"\nApplying to {query}: {answer}\n\n\\boxed{{{answer}}}"
    return answer, cot

# ── Equation Transformation (enhanced with more operations) ──
_EQ_OPERATIONS = {
    'addition': lambda a,b: str(a+b),
    'subtraction (A-B)': lambda a,b: str(a-b),
    'subtraction (B-A)': lambda a,b: str(b-a),
    'absolute difference': lambda a,b: str(abs(a-b)),
    'multiplication': lambda a,b: str(a*b),
    'concatenation (AB)': lambda a,b: str(a)+str(b),
    'concatenation (BA)': lambda a,b: str(b)+str(a),
    'integer division (A/B)': lambda a,b: str(a//b) if b!=0 else None,
    'integer division (B/A)': lambda a,b: str(b//a) if a!=0 else None,
    'modulo (A%B)': lambda a,b: str(a%b) if b!=0 else None,
    'modulo (B%A)': lambda a,b: str(b%a) if a!=0 else None,
    'XOR': lambda a,b: str(a^b),
    'bitwise AND': lambda a,b: str(a&b),
    'bitwise OR': lambda a,b: str(a|b),
    'max': lambda a,b: str(max(a,b)),
    'min': lambda a,b: str(min(a,b)),
    # NEW operations
    'digit sum': lambda a,b: str(sum(int(d) for d in str(a)) + sum(int(d) for d in str(b))),
    'digit product': lambda a,b: str(math.prod(int(d) for d in str(a)) * math.prod(int(d) for d in str(b))),
    'power (A^B)': lambda a,b: str(a**b) if b < 10 else None,
    'average floor': lambda a,b: str((a+b)//2),
}

def _parse_eq_examples(prompt):
    lines = prompt.strip().split('\n')
    examples, query = [], None
    for line in lines:
        line = line.strip()
        if 'determine the result for:' in line.lower():
            m = re.search(r'determine the result for:\s*(.+)', line, re.IGNORECASE)
            if m: query = m.group(1).strip()
        elif ' = ' in line and 'wonderland' not in line.lower() and 'transformation' not in line.lower():
            parts = line.split(' = ', 1)
            if len(parts) == 2:
                examples.append((parts[0].strip(), parts[1].strip()))
    return examples, query

def solve_equation_transformation(prompt, answer):
    examples, query = _parse_eq_examples(prompt)
    if not examples or not query or len(query) != 5: return None, None
    parsed = []
    for lhs, rhs in examples:
        if len(lhs) != 5: return None, None
        parsed.append((lhs[:2], lhs[2], lhs[3:], rhs))
    q_a, q_op, q_b = query[:2], query[2], query[3:]
    is_numeric = all(a.isdigit() and b.isdigit() for a,_,b,_ in parsed) and q_a.isdigit() and q_b.isdigit()
    if is_numeric:
        by_op = {}
        for a, op, b, rhs in parsed:
            by_op.setdefault(op, []).append((int(a), int(b), rhs))
        op_mapping = {}
        for op_char, op_examples in by_op.items():
            for op_name, op_func in _EQ_OPERATIONS.items():
                if all(op_func(a,b) == rhs for a,b,rhs in op_examples):
                    op_mapping[op_char] = op_name; break
        if q_op in op_mapping:
            predicted = _EQ_OPERATIONS[op_mapping[q_op]](int(q_a), int(q_b))
            if predicted == answer:
                cot = "Testing operators against examples:\n"
                for oc, on in op_mapping.items():
                    cot += f"  '{oc}' = {on}\n"
                cot += f"\n{q_a} '{q_op}' ({op_mapping[q_op]}) {q_b} = {predicted}\n\n\\boxed{{{predicted}}}"
                return predicted, cot
    # Scaffold
    cot = f"Equation transformation with {len(parsed)} examples.\n"
    for a,op,b,rhs in parsed[:4]:
        cot += f"  {a} '{op}' {b} = {rhs}\n"
    cot += f"\nFor {q_a} '{q_op}' {q_b}: {answer}\n\n\\boxed{{{answer}}}"
    return answer, cot

# ── Solver registry ──
SOLVER_MAP = {
    'Gravitational Constant': solve_gravity,
    'Unit Conversion': solve_unit_conversion,
    'Number Base Conversion': solve_base_conversion,
    'Text Encryption': solve_text_encryption,
    'Bit Manipulation': solve_bit_manipulation,
    'Equation Transformation': solve_equation_transformation,
}

# ── Enhanced fallback templates ──
FALLBACK_COT_TEMPLATES = {
    'Equation Transformation': (
        "I need to determine what each operator symbol represents.\n\n"
        "Each expression has format: operand1 (2 chars) + operator (1 char) + operand2 (2 chars) = result.\n"
        "I'll group examples by operator and test standard math operations.\n\n"
        "After systematic testing of addition, subtraction, multiplication, concatenation, XOR, and other "
        "operations against each operator group:\n\n"
        "\\boxed{{{answer}}}"
    ),
    'Bit Manipulation': (
        "I need to find the bit transformation by analyzing each output bit position.\n\n"
        "For each of the 8 output bits, I test if it can be explained as a function of input bits:\n"
        "- Direct copy or inversion of a single input bit\n"
        "- Two-bit operations: XOR, AND, OR, NAND, NOR, XNOR\n"
        "- Three-bit: majority vote, multiplexer/choice\n"
        "- Constant 0 or 1\n\n"
        "After analyzing all bit positions and applying the discovered transformation:\n\n"
        "\\boxed{{{answer}}}"
    ),
    'Number Base Conversion': (
        "I need to identify the numeral system conversion from the examples.\n\n"
        "Testing common systems: decimal↔Roman, binary↔decimal, octal, hexadecimal.\n"
        "After identifying the conversion rule and applying it:\n\n"
        "\\boxed{{{answer}}}"
    ),
    'Text Encryption': (
        "I need to build a letter substitution map from the examples.\n\n"
        "Each example provides character-to-character mappings.\n"
        "After building the complete substitution table and applying it:\n\n"
        "\\boxed{{{answer}}}"
    ),
    'Gravitational Constant': (
        "Using d = 0.5 * g * t^2, I can find g from each example.\n"
        "Then apply the discovered g to the query time value.\n\n"
        "\\boxed{{{answer}}}"
    ),
    'Unit Conversion': (
        "I'll find the conversion factor by dividing output by input for each example.\n"
        "Then multiply the query value by this factor.\n\n"
        "\\boxed{{{answer}}}"
    ),
}


# ═══════════════════════════════════════════════════════════════════════════════
# ADVANCED NEMOTRON v2 — Part 3: Data Formatting, Rewards, Training, Submission
# ═══════════════════════════════════════════════════════════════════════════════

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 9: DATA FORMATTING
# ════════════════════════════════════════════════════════════════

BOXED_INSTRUCTION = PROMPT_SUFFIX

if has_cot:
    if QUICK_TEST:
        TYPE_SAMPLES = {
            'Numeral Conversion': 10, 'Gravitational Constant': 10,
            'Unit Conversion': 10, 'Text Encryption': 10,
            'Bit Manipulation': 10, 'Equation Transformation': 10,
        }
    elif SMOKE_TEST:
        TYPE_SAMPLES = {
            'Numeral Conversion': 100, 'Gravitational Constant': 100,
            'Unit Conversion': 100, 'Text Encryption': 100,
            'Bit Manipulation': 100, 'Equation Transformation': 100,
        }
    else:
        # USE ALL VERIFIED-CORRECT SAMPLES (6558 total!)
        # konbu17 only used 2907 — we use everything for max coverage.
        TYPE_SAMPLES = {
            'Numeral Conversion': 9999,       # All 1491
            'Gravitational Constant': 9999,   # All 1511
            'Unit Conversion': 9999,          # All 1342
            'Text Encryption': 9999,          # All 1407
            'Bit Manipulation': 9999,         # All 607
            'Equation Transformation': 9999,  # All 200
        }

    sampled_dfs = []
    for ptype, n_samples in TYPE_SAMPLES.items():
        subset = cot_df[cot_df['type'] == ptype]
        if n_samples >= len(subset):
            sampled = subset
        else:
            sampled = subset.sample(n=n_samples, random_state=42)
        print(f'  {ptype}: {len(subset)} -> {len(sampled)}')
        sampled_dfs.append(sampled)

    train_pdf = pd.concat(sampled_dfs, ignore_index=True)
    train_pdf = train_pdf.sample(frac=1, random_state=42).reset_index(drop=True)
    print(f'\nTraining samples: {len(train_pdf)}')

    # Build SFT dataset from CoT
    records = []
    for _, row in train_pdf.iterrows():
        prompt = str(row['prompt'])
        answer = str(row['answer'])
        cot = str(row['generated_cot'])
        if not cot or cot == 'nan' or len(cot.strip()) < 5:
            continue
        cot_cleaned = re.sub(r'\\boxed\{[^}]*\}', '', cot).rstrip()
        user_content = prompt + PROMPT_SUFFIX
        assistant_content = cot_cleaned + f'\n</think>\n\\boxed{{{answer}}}'
        records.append({'messages': [
            {'role': 'user', 'content': user_content},
            {'role': 'assistant', 'content': assistant_content},
        ]})
    train_dataset = HFDataset.from_list(records)
    print(f'SFT records (CoT): {len(records)}')

else:
    def stratified_sample(df, n, seed=42):
        if n <= 0 or n >= len(df): return df.sample(fraction=1.0, seed=seed)
        type_counts = df.group_by('puzzle_type').agg(pl.len().alias('count'))
        total = len(df)
        samples = []
        remaining = n
        for row in type_counts.sort('count', descending=True).iter_rows(named=True):
            ptype = row['puzzle_type']
            pcount = row['count']
            alloc = max(1, round(n * pcount / total))
            alloc = min(alloc, pcount, remaining)
            subset = df.filter(pl.col('puzzle_type') == ptype).sample(n=alloc, seed=seed)
            samples.append(subset)
            remaining -= alloc
            if remaining <= 0: break
        return pl.concat(samples).sample(fraction=1.0, seed=seed)

    if SFT_SUBSAMPLE > 0:
        subsample = stratified_sample(train_full, SFT_SUBSAMPLE, seed=42)
    else:
        subsample = train_full.sample(fraction=1.0, seed=42)

    solver_stats = {'solved': 0, 'fallback': 0, 'total': 0,
                    'by_type': defaultdict(lambda: {'solved': 0, 'total': 0})}

    FALLBACK_COT_TEMPLATES = {
        'Equation Transformation': 'After testing operators:\n\n\\boxed{{{answer}}}',
        'Bit Manipulation': 'After analyzing bits:\n\n\\boxed{{{answer}}}',
        'Number Base Conversion': 'After converting:\n\n\\boxed{{{answer}}}',
        'Text Encryption': 'After substitution:\n\n\\boxed{{{answer}}}',
        'Gravitational Constant': 'Using d=0.5*g*t^2:\n\n\\boxed{{{answer}}}',
        'Unit Conversion': 'After conversion:\n\n\\boxed{{{answer}}}',
    }

    def build_training_text(example):
        prompt = example['prompt']
        answer = str(example['answer'])
        puzzle_type = example['puzzle_type']
        solver_stats['total'] += 1
        solver_stats['by_type'][puzzle_type]['total'] += 1
        assistant_msg = None
        if puzzle_type in SOLVER_MAP:
            predicted, cot = SOLVER_MAP[puzzle_type](prompt, answer)
            if cot is not None:
                assistant_msg = cot
                solver_stats['solved'] += 1
                solver_stats['by_type'][puzzle_type]['solved'] += 1
        if assistant_msg is None:
            solver_stats['fallback'] += 1
            template = FALLBACK_COT_TEMPLATES.get(puzzle_type, '\\boxed{{{answer}}}')
            assistant_msg = template.format(answer=answer)
        user_content = prompt + BOXED_INSTRUCTION
        try:
            messages = [
                {'role': 'user', 'content': user_content},
                {'role': 'assistant', 'content': assistant_msg},
            ]
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False,
                enable_thinking=True,
            )
        except Exception:
            text = f'<extra_id_1>User\n{user_content}\n<extra_id_1>Assistant\n{assistant_msg}\n'
        return {'text': text}

    hf_ds = HFDataset.from_pandas(subsample.to_pandas())
    train_dataset = hf_ds.map(build_training_text, remove_columns=hf_ds.column_names)
    print(f'\nSFT records (solver): {len(train_dataset)} | '
          f'Coverage: {solver_stats["solved"]/max(1,solver_stats["total"])*100:.0f}%')

# GRPO Dataset
if USE_GRPO:
    if not has_cot:
        grpo_sub = stratified_sample(train_full, GRPO_SUBSAMPLE, seed=99)
    else:
        grpo_sub = train_full.sample(n=min(GRPO_SUBSAMPLE, len(train_full)), seed=99)
    grpo_records = []
    for row in grpo_sub.iter_rows(named=True):
        grpo_records.append({
            'prompt': [{'role': 'user', 'content': row['prompt'] + BOXED_INSTRUCTION}],
            'ground_truth': str(row['answer']),
        })
    grpo_dataset = HFDataset.from_list(grpo_records)
    print(f'GRPO dataset: {len(grpo_dataset)} prompts x {GRPO_NUM_GENERATIONS} gens')
else:
    print('GRPO disabled — SFT only')

In [ ]:
# CELL 7: REWARD FUNCTIONS (ENHANCED)
# ════════════════════════════════════════════════════════════════════════════════

_reward_debug_counter = {"calls": 0}

def _normalize_answer(s):
    """Normalize answer for comparison with numeric tolerance."""
    s = s.strip()
    try:
        f = float(s)
        if f == int(f): return str(int(f))
        return str(f)
    except (ValueError, OverflowError):
        return s

def _answers_match(pred, gt, rel_tol=1e-2):
    """Match answers with numeric tolerance (competition uses 1e-2 relative)."""
    pred_n, gt_n = _normalize_answer(pred), _normalize_answer(gt)
    if pred_n == gt_n: return True
    # Case-insensitive for text answers
    if pred_n.lower() == gt_n.lower(): return True
    # Numeric tolerance
    try:
        p, g = float(pred), float(gt)
        if g == 0: return abs(p) < 1e-6
        return abs(p - g) / abs(g) <= rel_tol
    except (ValueError, OverflowError):
        return False

def _extract_boxed(content):
    match = re.search(r'\\boxed\{([^}]*)\}', content, re.DOTALL)
    if match: return match.group(1).strip()
    match = re.search(r'boxed\{([^}]*)\}', content, re.DOTALL)
    if match: return match.group(1).strip()
    return None

def _get_content(completion):
    if isinstance(completion, list):
        return completion[-1]["content"] if completion else ""
    return completion

def cosine_reward(completions, ground_truth, **kwargs):
    """Cosine-scaled accuracy reward with numeric tolerance."""
    max_len = GRPO_MAX_COMPLETION
    rewards = []
    _reward_debug_counter["calls"] += 1
    show_debug = _reward_debug_counter["calls"] <= 2

    for i, (completion, gt) in enumerate(zip(completions, ground_truth)):
        content = _get_content(completion)
        extracted = _extract_boxed(content)
        clen = len(content)
        progress = min(clen / max(max_len, 1), 1.0)
        cos_scale = 0.5 * (1.0 + math.cos(math.pi * progress))

        if extracted is not None and _answers_match(extracted, gt):
            reward = 0.1 + 0.9 * cos_scale  # [1.0, 0.1]
        elif extracted is not None:
            reward = -0.1 - 0.9 * (1.0 - cos_scale)  # [-0.1, -1.0]
        else:
            reward = -0.5 * progress

        rewards.append(reward)
        if show_debug and i < 2:
            tail = content[-100:] if len(content) > 100 else content
            print(f"  [COS] ext={extracted!r}, gt={gt!r}, r={reward:.3f}, len={clen}", flush=True)
    return rewards

def format_reward(completions, **kwargs):
    return [1.0 if _extract_boxed(_get_content(c)) is not None else 0.0 for c in completions]

def length_reward(completions, **kwargs):
    max_len = GRPO_MAX_COMPLETION
    return [-min(len(_get_content(c)) / max(max_len, 1), 1.0) for c in completions]

def reasoning_quality_reward(completions, **kwargs):
    """NEW: Rewards structured reasoning (step markers, examples cited)."""
    rewards = []
    for c in completions:
        content = _get_content(c)
        score = 0.0
        # Has step-by-step analysis
        if re.search(r'step\s*\d|example\s*\d|first|then|therefore|thus', content.lower()):
            score += 0.3
        # Shows work (equations, computations)
        if re.search(r'[=×÷+\-*/^]', content):
            score += 0.2
        # Isn't just the answer (has real reasoning, >100 chars before boxed)
        boxed_pos = content.find('\\boxed')
        if boxed_pos > 100:
            score += 0.2
        # Not degenerate (repetitive)
        words = content.split()
        if len(words) > 10:
            unique_ratio = len(set(words)) / len(words)
            if unique_ratio > 0.3:
                score += 0.3
        rewards.append(score)
    return rewards

# Sanity check
_reward_debug_counter["calls"] = 0
_test = cosine_reward(
    completions=["The answer is \\boxed{42}.", "No boxed.", "Wrong: \\boxed{99}." + " "*800],
    ground_truth=["42", "42", "42"],
)
print(f"Reward sanity: {[f'{r:.3f}' for r in _test]}  (expect ~[1.0, 0.0, ~-0.7])")
_reward_debug_counter["calls"] = 0

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 11: TRAINING CALLBACKS
# ════════════════════════════════════════════════════════════════

class PrintLossCallback(TrainerCallback):
    def __init__(self, phase='SFT'):
        self.phase = phase
        self.start_time = None
    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        print(f'\n{"="*60}\n[{self.phase}] Training started — {state.max_steps} steps\n{"="*60}', flush=True)
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs: return
        elapsed = time.time() - self.start_time if self.start_time else 0
        loss = logs.get('loss', logs.get('train_loss'))
        lr = logs.get('learning_rate')
        parts = [f'[{self.phase}] step {state.global_step}/{state.max_steps}']
        if loss is not None: parts.append(f'loss={loss:.4f}')
        if lr is not None: parts.append(f'lr={lr:.2e}')
        parts.append(f'elapsed={elapsed/60:.1f}min')
        print(' | '.join(parts), flush=True)
    def on_train_end(self, args, state, control, **kwargs):
        elapsed = time.time() - self.start_time if self.start_time else 0
        print(f'\n[{self.phase}] Complete — {state.global_step} steps in {elapsed/60:.1f}min', flush=True)


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 12: SFT TRAINING
# ════════════════════════════════════════════════════════════════

sft_kwargs = dict(
    output_dir=OUTPUT_DIR,
    num_train_epochs=SFT_EPOCHS,
    per_device_train_batch_size=SFT_BATCH,
    gradient_accumulation_steps=SFT_GRAD_ACCUM,
    learning_rate=SFT_LR,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    max_length=SFT_MAX_LEN,
    logging_steps=10,
    save_strategy='no',
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    dataloader_num_workers=2,
    remove_unused_columns=False,
    seed=42,
    report_to='none',
    packing=SFT_PACKING,
    neftune_noise_alpha=NEFTUNE_ALPHA,
)
if MAX_STEPS_SFT > 0:
    sft_kwargs['max_steps'] = MAX_STEPS_SFT

sft_args = SFTConfig(**sft_kwargs)

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    callbacks=[PrintLossCallback('SFT')],
)

est_steps = len(train_dataset) // (SFT_BATCH * SFT_GRAD_ACCUM) * SFT_EPOCHS
print(f'Starting SFT training... (~{est_steps} steps, {SFT_EPOCHS} epochs)')
t0 = time.time()
trainer.train()
elapsed = time.time() - t0
print(f'SFT done in {elapsed/60:.1f} min ({elapsed/3600:.1f} hours)')

# Save adapter
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Adapter saved to {OUTPUT_DIR}')

# Cleanup
del trainer; gc.collect(); torch.cuda.empty_cache()
print(f'GPU after SFT: {torch.cuda.memory_allocated()/1024**3:.1f} GB')
time.sleep(5)

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 13: GRPO TRAINING (OPTIMIZED)
# ════════════════════════════════════════════════════════════════

if USE_GRPO:
    grpo_kwargs = dict(
        output_dir=OUTPUT_DIR + '_grpo',
        num_train_epochs=GRPO_EPOCHS,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=2,        # Was 4 — fewer = faster steps
        learning_rate=GRPO_LR,
        lr_scheduler_type='cosine',
        warmup_steps=5,
        num_generations=GRPO_NUM_GENERATIONS,
        generation_batch_size=GRPO_NUM_GENERATIONS,
        max_completion_length=GRPO_MAX_COMPLETION,
        temperature=GRPO_TEMPERATURE,
        beta=GRPO_BETA,
        logging_steps=1 if QUICK_TEST else 5,
        save_strategy='no',
        bf16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        report_to='none',
        max_grad_norm=0.1,
    )
    if MAX_STEPS_GRPO > 0:
        grpo_kwargs['max_steps'] = MAX_STEPS_GRPO

    grpo_config = GRPOConfig(**grpo_kwargs)

    grpo_trainer = GRPOTrainer(
        model=model,
        args=grpo_config,
        train_dataset=grpo_dataset,
        processing_class=tokenizer,
        reward_funcs=[
            cosine_reward,
            format_reward,
            reasoning_quality_reward,
        ],
        callbacks=[PrintLossCallback('GRPO')],
    )

    print(f'Starting GRPO training... (max_steps={MAX_STEPS_GRPO}, max_completion={GRPO_MAX_COMPLETION})')
    t0 = time.time()
    grpo_trainer.train()
    elapsed = time.time() - t0
    print(f'GRPO done in {elapsed/60:.1f} min')

    # Save final adapter
    model.save_pretrained(OUTPUT_DIR)
    del grpo_trainer; gc.collect(); torch.cuda.empty_cache()
    print('GRPO complete!')
else:
    print('GRPO skipped (USE_GRPO=False)')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 14: PACKAGE SUBMISSION
# ════════════════════════════════════════════════════════════════

SUBMISSION_DIR = '/kaggle/working/submission_adapter'
os.makedirs(SUBMISSION_DIR, exist_ok=True)

# Copy adapter files
required_files = ['adapter_config.json', 'adapter_model.safetensors']
for fname in required_files:
    src = os.path.join(OUTPUT_DIR, fname)
    dst = os.path.join(SUBMISSION_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'  {fname}: {os.path.getsize(dst)/1024/1024:.1f} MB')
    else:
        print(f'  ⚠️ {fname} not found!')

# Fix adapter_config for submission
config_path = os.path.join(SUBMISSION_DIR, 'adapter_config.json')
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        cfg = json.load(f)
    cfg['base_model_name_or_path'] = BASE_MODEL_NAME
    cfg['inference_mode'] = True
    cfg['lora_dropout'] = 0.0
    with open(config_path, 'w') as f:
        json.dump(cfg, f, indent=2)

# Create submission.zip
zip_path = '/kaggle/working/submission.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        fpath = os.path.join(SUBMISSION_DIR, fname)
        if os.path.exists(fpath):
            zf.write(fpath, fname)
            print(f'  Added {fname}')

zip_sz = os.path.getsize(zip_path) / 1024 / 1024
print(f'\n✅ submission.zip: {zip_sz:.1f} MB')
print('Done! Ready to submit.')
